# ReportLens — OCR demo (Tesseract)

Runs the **production OCR pipeline** on synthetic lab reports and reports accuracy (CER).
Engine is **Tesseract**: mature, CPU-only, no training, and highly accurate on clean
printed reports.

**No GPU needed.** Just *Runtime → Run all*. Takes about a minute.

(The from-scratch TrOCR fine-tuning pipeline lives in `train_ocr_colab.ipynb` as an ML
artifact; this notebook is the one that actually powers the product.)

## 1. Get the code + dependencies

In [ ]:
import os
%cd /content
if os.path.isdir('reportlens'):
    !cd reportlens && git fetch -q origin && git reset -q --hard origin/main
else:
    !git clone -q https://github.com/Satwiksingh123/reportlens-backend-.git reportlens
%cd /content/reportlens
print('--- running commit ---'); !git log --oneline -1
# tesseract binary is pre-installed on Colab; install pytesseract + Pillow
!pip install -q -e "services/ocr_engine[ocr]" pillow
!tesseract --version | head -1

## 2. Run the pipeline on one report
Prints the ground truth next to the OCR output and a character-accuracy figure.

In [ ]:
%cd /content/reportlens/services/ocr_engine
!python -m ocr_engine.sanity_check --engine tesseract --seed 9999

## 3. Average accuracy over many reports
A single page can be lucky/unlucky; this averages CER across 30 random reports for a
trustworthy number.

In [ ]:
import sys; sys.path.insert(0, '../data_synthesis')
from data_synthesis.generator import generate_report, render_report
from ocr_engine.recognizer import TesseractRecognizer
from ocr_engine.infer import extract_text_from_pil
from ocr_engine.sanity_check import _cer

rec = TesseractRecognizer()
cers = []
for s in range(30):
    rep = generate_report(seed=1000 + s)
    img, _ = render_report(rep, add_noise=True, seed=1000 + s)
    gt = ' '.join(' '.join(l for l in rep.text_lines if l.strip()).split())
    hyp = ' '.join(extract_text_from_pil(img, rec).split())
    cers.append(_cer(gt, hyp))
avg = sum(cers) / len(cers)
print(f'average CER over {len(cers)} reports: {avg:.3f}')
print(f'=> character accuracy ~ {(1 - avg) * 100:.1f}%')